# Ch 18 · Collision IV: Heightfields, SDFs and User SDF Plugins

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/18-heightfields-and-sdfs.ipynb)

Companion notebook for the chapter page `chapters/18-heightfields-and-sdfs.html` ([page](../chapters/18-heightfields-and-sdfs.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

**Reading order.** The cells follow the chapter page section by section. The example numbers are not sequential (18.1, 18.3, 18.4, 18.2, 18.7, 18.5, 18.6, 18.8, 18.9) because they keep the file names used on the page (`ex2_...py` is Example 18.2), so each cell header matches the page's run notes.

## 1. Heightfields: the TERRAIN scene
Every geom that touches a heightfield goes through the HFIELD prism kernel (`collision_convex.py::ccd_hfield_kernel_builder`), even a sphere.

In [ ]:
# Example 18.1 - ex1_terrain.py
"""Example 18.1 - the TERRAIN scene: which collision path makes each contact.

TERRAIN (course model M7) has a 5 x 5 heightfield and four free bodies: a sphere, an ellipsoid,
a cylinder and a tetrahedron mesh. We step 2 worlds for 100 steps and then sort every contact
by the body it touches and by the narrowphase path that produced it. Every geom here collides
with the hfield, so every contact comes from the HFIELD prism kernel
(collision_convex.py::ccd_hfield_kernel_builder), even for the sphere.
Run: python ex1_terrain.py
"""
import collections

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.collision_driver import MJ_COLLISION_TABLE  # private API

np.set_printoptions(precision=4, suppress=True)

TERRAIN = """
<mujoco model="terrain">
  <option timestep="0.005"/>
  <asset>
    <hfield name="bumps" nrow="5" ncol="5" size="1 1 .2 .1"
      elevation="0 0 0 0 0  0 .5 1 .5 0  0 1 0 1 0  0 .5 1 .5 0  0 0 0 0 0"/>
    <mesh name="tetra" vertex="0 0 0  .2 0 0  0 .2 0  0 0 .2"/>
  </asset>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="ground" type="hfield" hfield="bumps"/>
    <body name="ball" pos="0.2 0.1 .6"><freejoint/><geom type="sphere" size=".08"/></body>
    <body name="ellipsoid" pos="-0.3 0.2 .6"><freejoint/>
      <geom type="ellipsoid" size=".1 .07 .05"/></body>
    <body name="cyl" pos="0.3 -0.3 .6"><freejoint/><geom type="cylinder" size=".06 .05"/></body>
    <body name="tetra" pos="-0.3 -0.3 .6"><freejoint/><geom type="mesh" mesh="tetra"/></body>
  </worldbody>
</mujoco>
"""

NWORLD = 2
NSTEP = 100
G = mjw.GeomType


def collision_path(t1, t2):
    """Name the narrowphase path for a (lower type, higher type) geom pair."""
    if G.HFIELD in (t1, t2):
        return "HFIELD (prisms + GJK/EPA)"
    if G.SDF in (t1, t2):
        return "SDF (gradient descent)"
    return MJ_COLLISION_TABLE[(t1, t2)].name


mjm = mujoco.MjModel.from_xml_string(TERRAIN)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
print(f"nq {mjm.nq}  nv {mjm.nv}  ngeom {mjm.ngeom}  nhfield {mjm.nhfield}  "
      f"hfield {mjm.hfield_nrow[0]}x{mjm.hfield_ncol[0]}  size {mjm.hfield_size[0]}")
print(f"naconmax {d.naconmax}  njmax {d.njmax}")

for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()

nacon = min(int(d.nacon.numpy()[0]), d.naconmax)
geom = d.contact.geom.numpy()[:nacon]
world = d.contact.worldid.numpy()[:nacon]
dist = d.contact.dist.numpy()[:nacon]
per_body = collections.Counter()
per_path = collections.Counter()
for (g1, g2), w in zip(geom, world):
    body = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_BODY, int(mjm.geom_bodyid[g2]))
    per_body[(int(w), body)] += 1
    per_path[collision_path(G(int(mjm.geom_type[g1])), G(int(mjm.geom_type[g2])))] += 1

print(f"after {NSTEP} steps: nacon = {nacon} over {NWORLD} worlds")
for (w, body), n in sorted(per_body.items()):
    print(f"  world {w}  {body:10s} {n} contact(s)")
print("contacts per collision path:", dict(per_path))
print("min / max contact dist:", round(float(dist.min()), 4), round(float(dist.max()), 4))
print("body heights world 0 (z):", d.xpos.numpy()[0, 1:, 2])
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 2. Four contacts, not six
MJWarp keeps at most 4 contacts per heightfield pair; MuJoCo C keeps its own selection. Compare physics outcomes, not contact counts.

In [ ]:
# Example 18.3 - ex3_hfield_vs_c.py
"""Example 18.3 - one sphere on a heightfield: MJWarp keeps at most 4 contacts, MuJoCo C more.

A sphere of radius 0.1 sinks 5 mm into a flat 11 x 11 heightfield (cells 0.1 m wide). Both
engines run GJK/EPA on every prism under the sphere and collect the touching ones. Then MJWarp's
ccd_hfield_kernel keeps at most 4 of them (the deepest, the farthest from it, the farthest from
that line, the farthest from the triangle edge) while MuJoCo C keeps its own selection.
Pose A puts the sphere exactly on a grid vertex: the six prisms around the vertex all report the
same point. Pose B moves it off the vertex. We call mj_forward and mjw.forward on the same pose,
print both contact lists, then step both engines for 1 s and compare the resting height.
Run: python ex3_hfield_vs_c.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

N = 11
FLAT = " ".join(["0"] * (N * N))
XML = f"""
<mujoco model="sphere_on_hfield">
  <option timestep="0.005"/>
  <asset>
    <hfield name="flat" nrow="{N}" ncol="{N}" size="0.5 0.5 0.1 0.05" elevation="{FLAT}"/>
  </asset>
  <worldbody>
    <geom name="ground" type="hfield" hfield="flat"/>
    <body name="ball" pos="0 0 0.095"><freejoint/>
      <geom name="ball" type="sphere" size="0.1"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=16, njmax=64)
mjd = mujoco.MjData(mjm)


def show_c(mjd):
    print(f"  MuJoCo C : ncon  = {mjd.ncon}")
    for i in range(mjd.ncon):
        c = mjd.contact[i]
        normal = np.round(c.frame[:3], 3)
        print(f"    pos {np.round(c.pos, 4)}  dist {c.dist:+.4f}  normal {normal}")


def show_warp(d):
    nacon = min(int(d.nacon.numpy()[0]), d.naconmax)
    print(f"  MJWarp   : nacon = {nacon}")
    pos = d.contact.pos.numpy()[:nacon]
    dist = d.contact.dist.numpy()[:nacon]
    frame = d.contact.frame.numpy()[:nacon]
    for i in np.argsort(dist):
        print(f"    pos {pos[i]}  dist {dist[i]:+.4f}  normal {frame[i, 0]}")


for label, xy in [("A: on a grid vertex", (0.0, 0.0)), ("B: off the vertex", (0.03, 0.02))]:
    qpos = mjm.qpos0.copy()
    qpos[:3] = [xy[0], xy[1], 0.095]
    mjd.qpos[:] = qpos
    mujoco.mj_forward(mjm, mjd)
    d.qpos.assign(qpos[None].astype(np.float32))
    mjw.forward(m, d)
    print(f"pose {label}, ball center {qpos[:3]}")
    show_c(mjd)
    show_warp(d)

# Different contact sets, same physics: let both settle from pose B for 1 s.
mujoco.mj_resetData(mjm, mjd)
mjd.qpos[:3] = [0.03, 0.02, 0.15]
d.qpos.assign(mjd.qpos[None].astype(np.float32))
d.qvel.zero_()
for _ in range(200):
    mujoco.mj_step(mjm, mjd)
    mjw.step(m, d)
print(f"after 1 s: ball z  C {mjd.qpos[2]:.5f}   MJWarp {float(d.qpos.numpy()[0, 2]):.5f}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 3. HFIELD overflow
At most `MJ_MAXCONPAIR = 50` touching prisms per pair are collected. The device warning lines are printed by the kernel (on CPU they can appear after the Python lines).

In [ ]:
# Example 18.4 - ex4_hfield_overflow.py
"""Example 18.4 - raise the heightfield resolution until the HFIELD overflow bit appears.

A 0.2 m box rests flat on a 1 m x 1 m heightfield with a 2 mm overlap. Every prism under the
box touches it, so the number of candidate prism contacts grows with the square of the
resolution. ccd_hfield_kernel stores at most MJ_MAXCONPAIR candidates per geom pair; when the
buffer is full and another prism remains it stops, sets OverflowType.HFIELD for that world and
prints a warning (once per pair, from the device). The rows are scanned from low y to high y, so
an overflowed pair only "sees" the lower part of the footprint: look at the y range of the
written contacts. The fix is a coarser grid (or a smaller colliding geom). "sub-grid prisms" is
the number of prisms the kernel visits (same index formulas); only touching ones fill the buffer.
Run: python ex4_hfield_overflow.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def scene(n):
    flat = " ".join(["0"] * (n * n))
    return f"""
<mujoco model="box_on_hfield">
  <asset><hfield name="flat" nrow="{n}" ncol="{n}" size="0.5 0.5 0.1 0.05"
    elevation="{flat}"/></asset>
  <worldbody>
    <geom name="ground" type="hfield" hfield="flat"/>
    <body name="crate" pos="0.013 0.007 0.098"><freejoint/>
      <geom name="crate" type="box" size="0.1 0.1 0.1"/></body>
  </worldbody>
</mujoco>"""


def subgrid_prisms(n, size=0.5, x=(0.013 - 0.1, 0.013 + 0.1), y=(0.007 - 0.1, 0.007 + 0.1)):
    """Prisms the kernel visits: the same cmin/cmax/rmin/rmax formulas as ccd_hfield_kernel."""
    scale = 0.5 * (n - 1) / size
    cmin = max(0, int(np.floor((x[0] + size) * scale)))
    cmax = min(n - 1, int(np.ceil((x[1] + size) * scale)))
    rmin = max(0, int(np.floor((y[0] + size) * scale)))
    rmax = min(n - 1, int(np.ceil((y[1] + size) * scale)))
    return 2 * (cmax - cmin) * (rmax - rmin)    # rows r in [rmin, rmax), cells c in (cmin, cmax]


print("MJ_MAXCONPAIR =", mujoco.mjMAXCONPAIR, "| box footprint y range: [-0.093, 0.107]")
rows = []
for n in [11, 16, 21, 26, 31]:
    mjm = mujoco.MjModel.from_xml_string(scene(n))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=1, nconmax=16, njmax=64)
    mjw.forward(m, d)
    wp.synchronize()
    nacon = min(int(d.nacon.numpy()[0]), d.naconmax)
    y = d.contact.pos.numpy()[:nacon, 1]
    bits = int(d.overflow.numpy()[0])
    names = [b.name for b in mjw.OverflowType if bits & b] or ["none"]
    cell = 1.0 / (n - 1)
    nprism = subgrid_prisms(n)
    print(f"n = {n:2d} (cell {cell * 100:4.1f} cm, sub-grid {nprism:3d} prisms): "
          f"nacon {nacon}, contact y in [{y.min():+.3f}, {y.max():+.3f}], "
          f"overflow {'|'.join(names)}")
    rows.append({"n": n, "cell": round(cell, 4), "prisms": nprism, "nacon": nacon,
                 "ymin": round(float(y.min()), 3), "ymax": round(float(y.max()), 3),
                 "overflow": bits})

ok = [r["n"] for r in rows if r["overflow"] == 0]
print(f"fix: use at most {max(ok)} x {max(ok)} rows/cols under a geom this size "
      f"(or a smaller geom); finer grids drop part of the footprint.")
print("to silence only the print (the bit is still set): "
      "m.opt.warn_overflow &= ~mjw.OverflowType.HFIELD")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"maxconpair": mujoco.mjMAXCONPAIR, "rows": rows}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 4. Build terrain with MjSpec, and the default buffer heuristic
`io.py::_default_nconmax` adds `nv * 0.35 * 10` contacts per world when the model has a heightfield. We print the default, then use explicit, smaller sizes.

In [ ]:
# Example 18.2 - ex2_hfield_spec.py
"""Example 18.2 - build a 32 x 32 random-noise heightfield with MjSpec and drop 16 balls on it.

The terrain is smoothed random noise in [0, 1] (MuJoCo scales it by size[2] = 0.3 m). Each world
drops 16 balls from its own random (x, y) grid, so the worlds end in different places. We print
the default buffer sizes that io.py's heuristic picks for a model with an hfield, the explicit
sizes we use instead, a histogram of the final ball heights, the contact and row counts (at the
end and the peak over the run) and the overflow status.
Run: python ex2_hfield_spec.py [--dump-json PATH]
  --dump-json PATH writes the elevation grid and the world-0 ball tracks for the page figure.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 300
NROW = NCOL = 32
NBALL = 16
SIZE = [1.0, 1.0, 0.3, 0.1]          # x half-size, y half-size, max elevation, base depth


def noise_terrain(nrow, ncol, seed=0):
    """Random noise, smoothed by a 5x5 box filter (with wrap-around), rescaled to [0, 1]."""
    rng = np.random.default_rng(seed)
    z = rng.random((nrow, ncol))
    k = 2
    z = sum(np.roll(np.roll(z, i, 0), j, 1) for i in range(-k, k + 1) for j in range(-k, k + 1))
    return (z - z.min()) / (z.max() - z.min())


def build(elev):
    spec = mujoco.MjSpec()
    spec.option.timestep = 0.005
    hf = spec.add_hfield(name="noise", nrow=NROW, ncol=NCOL, size=SIZE)
    hf.userdata = elev.ravel().tolist()     # row-major: row r is y, column c is x
    spec.worldbody.add_light(pos=[0, 0, 3], dir=[0, 0, -1])
    spec.worldbody.add_geom(name="terrain", type=mujoco.mjtGeom.mjGEOM_HFIELD, hfieldname="noise")
    for i in range(NBALL):
        b = spec.worldbody.add_body(name=f"ball{i}", pos=[0, 0, 0.6 + 0.02 * i])
        b.add_freejoint()
        b.add_geom(type=mujoco.mjtGeom.mjGEOM_SPHERE, size=[0.05, 0, 0])
    return spec.compile()


elev = noise_terrain(NROW, NCOL)
mjm = build(elev)
m = mjw.put_model(mjm)
print(f"hfield {mjm.hfield_nrow[0]}x{mjm.hfield_ncol[0]}, nbody {mjm.nbody}, nv {mjm.nv}, "
      f"ngeom {mjm.ngeom}")

d_default = mjw.make_data(mjm, nworld=NWORLD)
print(f"default sizes (io.py heuristic): nconmax {d_default.naconmax // NWORLD} per world, "
      f"njmax {d_default.njmax}")
del d_default
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=96, njmax=384)
print(f"explicit sizes: nconmax 96 per world (naconmax {d.naconmax}), njmax {d.njmax}")

# per-world drop positions: a jittered 4 x 4 grid inside the terrain
rng = np.random.default_rng(1)
qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
ticks = np.linspace(-0.6, 0.6, 4)
grid = np.stack(np.meshgrid(ticks, ticks), -1).reshape(-1, 2)
for w in range(NWORLD):
    xy = grid + rng.uniform(-0.12, 0.12, size=grid.shape)
    for i in range(NBALL):
        qpos[w, 7 * i:7 * i + 2] = xy[i]
d.qpos.assign(qpos)

tracks = []
peak_nacon = peak_nefc = 0          # running maxima (Ch 9 recipe): the landing needs the most
for step in range(NSTEP):
    mjw.step(m, d)
    peak_nacon = max(peak_nacon, int(d.nacon.numpy()[0]))       # host reads: sizing run only
    peak_nefc = max(peak_nefc, int(d.nefc.numpy().max()))
    if args.dump_json and step % 10 == 0:       # host reads for the figure only
        tracks.append(d.qpos.numpy()[0].reshape(NBALL, 7)[:, :3].astype(float).round(3).tolist())
wp.synchronize()

z = d.qpos.numpy().reshape(NWORLD, NBALL, 7)[:, :, 2].ravel()
counts, edges = np.histogram(z, bins=6, range=(0.0, 0.42))
off = int((z < 0.0).sum())
print(f"final ball heights over {NWORLD} worlds x {NBALL} balls "
      f"({off} rolled off the terrain edge and fell, z < 0):")
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"  z in [{lo:.2f}, {hi:.2f}): {c:3d} " + "#" * int(c // 2))
nacon = int(d.nacon.numpy()[0])
print(f"at the end (balls at rest): nacon {nacon} of naconmax {d.naconmax}; "
      f"nefc per world max {int(d.nefc.numpy().max())}")
print(f"peak over all {NSTEP} steps:  nacon {peak_nacon}; nefc per world max {peak_nefc} of njmax {d.njmax}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nrow": NROW, "ncol": NCOL, "size": SIZE, "dt": 10 * mjm.opt.timestep,
                   "elev": elev.round(3).tolist(), "tracks": tracks, "radius": 0.05,
                   "hist": {"counts": counts.tolist(), "edges": edges.round(3).tolist()}},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Optional plot: the terrain and where the world-0 balls ended (numbers are printed above).
try:
    import matplotlib.pyplot as plt
    plt.imshow(elev, origin='lower', extent=[-1, 1, -1, 1], cmap='terrain')
    xy = d.qpos.numpy()[0].reshape(NBALL, 7)[:, :2]
    plt.scatter(xy[:, 0], xy[:, 1], c='k', s=12)
    plt.title('noise terrain and final world-0 ball positions'); plt.show()
except ImportError:
    print('skipped: matplotlib not installed')

## 5. `sdf_initpoints` and `sdf_iterations`
A mesh with `type="sdf"` and no plugin is sampled from an octree, so no registration is needed and this runs in the notebook process. Both options are kernel inputs: changing them does not recompile.

In [ ]:
# Example 18.7 - ex7_sdf_iterations.py
"""Example 18.7 - sweep sdf_initpoints and sdf_iterations on one SDF contact.

The SDF geom is a 0.2 m cube mesh with type="sdf" and no plugin: MuJoCo compiles the mesh into
an octree of SDF samples, and MJWarp reads it with collision_sdf.py::sample_volume_sdf (no user
code, so no registration is needed and this runs in-process). The cube sits on a plane with a
known overlap of 1 mm. The SDF path reports dist = max(A, B) at the point where the descent
stops, so a flat overlap of depth h gives at best -h/2 (here -0.0005), not -h.
For each setting we call mjw.collision once and print the number of contacts (every starting
point that ends in contact writes its own contact), the min and mean distance, and how much of
the 0.2 x 0.2 m bottom face the contacts cover. Both options are plain kernel inputs, so
changing them between calls does not recompile anything.
Run: python ex7_sdf_iterations.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

CUBE = "-1 -1 -1  1 -1 -1  1 1 -1  -1 1 -1  -1 -1 1  1 -1 1  1 1 1  -1 1 1"
XML = f"""
<mujoco model="sdf_cube_on_plane">
  <asset><mesh name="cube" vertex="{CUBE}" scale=".1 .1 .1"/></asset>
  <worldbody>
    <geom name="floor" type="plane" size="1 1 .1"/>
    <body name="cube" pos="0.013 -0.021 0.099" euler="0 0 17"><freejoint/>
      <geom name="cube" type="sdf" mesh="cube"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(XML)
print(f"geom types {[mujoco.mjtGeom(t).name for t in mjm.geom_type]}, mesh_octadr "
      f"{mjm.mesh_octadr.tolist()}, octree nodes {mjm.noct}")
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=256, njmax=1024)
mjw.kinematics(m, d)

rows = []
print(" initpoints  iterations  nacon   min dist   mean dist   face coverage (x, y extent)")
for initpoints, iterations in [(1, 10), (5, 10), (10, 10), (40, 10), (100, 10),
                               (40, 0), (40, 1), (40, 3), (40, 30)]:
    m.opt.sdf_initpoints = initpoints
    m.opt.sdf_iterations = iterations
    mjw.collision(m, d)
    wp.synchronize()
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    dist = d.contact.dist.numpy()[:n]
    pos = d.contact.pos.numpy()[:n]
    ext = np.ptp(pos[:, :2], axis=0) if n else np.zeros(2)
    dmin = float(dist.min()) if n else float("nan")
    dmean = float(dist.mean()) if n else float("nan")
    print(f" {initpoints:10d}  {iterations:10d}  {n:5d}   {dmin:+.5f}   {dmean:+.5f}    {ext}")
    rows.append({"initpoints": initpoints, "iterations": iterations, "nacon": n,
                 "min": round(dmin, 5), "mean": round(dmean, 5),
                 "xy": pos[:, :2].astype(float).round(4).tolist()})
print("overlap 0.00100 -> best possible max(A, B) = -0.00050; overflow:",
      "none" if not d.overflow.numpy().any() else d.overflow.numpy())
mjd = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, mjd)
print(f"MuJoCo C, same pose: ncon {mjd.ncon}, min dist {min(c.dist for c in mjd.contact):+.5f} "
      "(C uses a different collider for plane-SDF)")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"rows": rows, "half": 0.1, "yaw_deg": 17, "center": [0.013, -0.021]}, f,
                  separators=(",", ":"))
    print("wrote", args.dump_json)

## Why some examples run as `!python file.py`
MJWarp's SDF kernel calls `collision_sdf.user_sdf` / `user_sdf_grad`. Warp builds that kernel the first time it launches **in this process**, using whatever function those names point to at that moment. A patch assigned later is ignored by the already-built kernel (Example 18.6 shows it: `nacon` stays 0).

So every example that registers or patches an SDF runs in a **fresh process**: a `%%writefile` cell puts the script on disk and `!python` runs it. Alternative: put the patch in the very first code cell after setup, or use *Runtime > Restart session* and run the patching cell before anything else.

## 6. SDF plugins without registration
The script runs its work in a child process and counts the placeholder ERROR lines.

In [ ]:
%%writefile ex5_sdf_unregistered.py
"""Example 18.5 - an SDF plugin model WITHOUT registration: C finds contacts, MJWarp finds none.

nutbolt.xml (shipped in the wheel) uses the MuJoCo C plugins mujoco.sdf.nut and mujoco.sdf.bolt.
MuJoCo C evaluates them natively. MJWarp cannot run C plugin code inside a Warp kernel: it calls
collision_sdf.user_sdf / user_sdf_grad, which are placeholders that wp.printf an ERROR line and
return 0. With a distance of exactly 0 everywhere, no contact passes the margin test.

Fresh process: the work runs in a child process (this file with --child) so that its device
printf lines (on stdout) can be captured and counted instead of flooding the output. The child
writes its own report to stderr, so the two streams cannot interleave mid-line; the parent prints
the report and a count of each distinct ERROR line. In a notebook run it with
  !python ex5_sdf_unregistered.py
Run: python ex5_sdf_unregistered.py
"""
import collections
import subprocess
import sys

if "--child" not in sys.argv:
    run = subprocess.run([sys.executable, "-I", __file__, "--child"], capture_output=True,
                         text=True)
    for line in run.stderr.splitlines():
        if "CUDA" not in line:                    # skip Warp's "no CUDA driver" notice on CPU
            print(line)
    errors = collections.Counter(l for l in run.stdout.splitlines() if l.strip())
    for line, n in sorted(errors.items()):
        print(f"{n:5d} x  {line}")
    sys.exit(0)

# ---------------- child process ----------------
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath


def report(*a):
    print(*a, file=sys.stderr)


NUTBOLT = epath.resource_path("mujoco_warp") / "test_data/collision_sdf/nutbolt.xml"

mjm = mujoco.MjModel.from_xml_path(str(NUTBOLT))
mjd = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, mjd)
report(f"geom types {[mujoco.mjtGeom(t).name for t in mjm.geom_type]}, "
       f"geom_plugin {mjm.geom_plugin.tolist()}, plugin types {mjm.plugin.tolist()}")
report(f"MuJoCo C : ncon  = {mjd.ncon}")

m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)          # default sizes: an SDF model gets nconmax >= 64
report(f"default naconmax {d.naconmax}, sdf_initpoints {m.opt.sdf_initpoints}, "
       f"sdf_iterations {m.opt.sdf_iterations}")
mjw.forward(m, d)
wp.synchronize()
report(f"MJWarp   : nacon = {int(d.nacon.numpy()[0])}  (user_sdf was never registered)")

In [ ]:
!python ex5_sdf_unregistered.py

## 7. Register first, then collide
`register_sdf_plugins(mjw)` (shipped test helper, private API) patches Warp versions of nut, bolt, torus and gear. The script also starts a `--late` child that registers too late.

In [ ]:
%%writefile ex6_sdf_registered.py
"""Example 18.6 - register Warp versions of the SDF plugins FIRST, then nutbolt.xml collides.

register_sdf_plugins(mjw) (a helper shipped in mujoco_warp.test_data) builds @wp.func versions
of user_sdf / user_sdf_grad that dispatch on the plugin type id (nut, bolt, torus, gear) and
assigns them to mujoco_warp._src.collision_sdf. Warp reads those module globals when it compiles
the SDF kernel, so the patch must come before the first put_model / forward / step in the
process. Fresh process: run this file as its own process (in a notebook: !python ex6_...py).

This file also starts a second child process with --late that does it in the WRONG order: one
forward with the placeholders, then register, then forward again. The second forward still has
nacon 0, because the compiled SDF kernel already holds the placeholder.
Run: python ex6_sdf_registered.py
"""
import subprocess
import sys

LATE = "--late" in sys.argv

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp.test_data.collision_sdf.utils import register_sdf_plugins  # private API
from etils import epath

np.set_printoptions(precision=4, suppress=True)

if not LATE:
    sdf_types = register_sdf_plugins(mjw)      # BEFORE any MJWarp kernel is compiled
    print("registered plugin type ids:", {k.name: v for k, v in sdf_types.items()})

NUTBOLT = epath.resource_path("mujoco_warp") / "test_data/collision_sdf/nutbolt.xml"
mjm = mujoco.MjModel.from_xml_path(str(NUTBOLT))
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)


def nacon_after_forward():
    mjw.forward(m, d)
    wp.synchronize()
    return int(d.nacon.numpy()[0])


if LATE:
    n0 = nacon_after_forward()                 # compiles the SDF kernel with the placeholders
    register_sdf_plugins(mjw)                  # too late for this process
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=1)
    n1 = nacon_after_forward()
    print(f"late registration: nacon before {n0}, after register + new put_model {n1}",
          file=sys.stderr)
    sys.exit(0)

mjd = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, mjd)
nacon = nacon_after_forward()
print(f"opt.sdf_initpoints {m.opt.sdf_initpoints}, opt.sdf_iterations {m.opt.sdf_iterations}")
attr = m.plugin_attr.numpy()
print(f"plugin_attr (first 2 of {attr.shape[1]} slots): "
      f"{attr[:, :2].astype(float).round(4).tolist()}")
print(f"geom_plugin_index {m.geom_plugin_index.numpy().tolist()}, "
      f"m.plugin {m.plugin.numpy().tolist()}")
print(f"MuJoCo C : ncon  = {mjd.ncon}")
print(f"MJWarp   : nacon = {nacon}")
g = d.contact.geom.numpy()[:nacon]
dist = d.contact.dist.numpy()[:nacon]
names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, i) for i in range(mjm.ngeom)]
for (a, b), r in zip(g, dist):
    print(f"  {names[a]:>6s} - {names[b]:<5s} dist {r:+.5f}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

late = subprocess.run([sys.executable, "-I", __file__, "--late"], capture_output=True, text=True)
for line in late.stderr.splitlines():
    if "CUDA" not in line:
        print(line)
print("placeholder ERROR lines printed by the late process:",
      sum(l.startswith("ERROR") for l in late.stdout.splitlines()))

In [ ]:
!python ex6_sdf_registered.py

## 8. Your own analytic SDF
A rounded tile patched into `user_sdf` / `user_sdf_grad`, reusing the `mujoco.sdf.torus` attribute layout. Fresh process again.

In [ ]:
%%writefile ex8_custom_sdf.py
"""Example 18.8 - write your own analytic SDF and patch it in, the way register_sdf_plugins does.

Fresh process: the patch must happen before the first MJWarp collision kernel compiles in this
process (in a notebook: !python ex8_custom_sdf.py).

The shape is a "rounded tile": a flat square of half-size R (the core box has half-sizes
(R, R, 0)) inflated by a rounding radius r. Its bounding box, (R + r, R + r, r), is exactly the
bounding box of a torus with radius1 = R and radius2 = r, so we reuse the attribute layout of
the mujoco.sdf.torus plugin: attr[0] = R, attr[1] = r. MuJoCo C (and its visual mesh) still
see a torus; only MJWarp, through our patched user_sdf / user_sdf_grad, sees the tile.
We drop a ball on the centre: on a torus it falls through the hole, on the tile it rests at
z = r + ball radius.
Run: python ex8_custom_sdf.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec_pluginattr  # private API

np.set_printoptions(precision=4, suppress=True)

# 1. The plugin type id of mujoco.sdf.torus in this process (what geom -> m.plugin stores).
_probe = mujoco.MjModel.from_xml_string("""<mujoco><extension>
  <plugin plugin="mujoco.sdf.torus"><instance name="t"/></plugin></extension>
  <asset><mesh name="tm"><plugin instance="t"/></mesh></asset>
  <worldbody><geom type="sdf" mesh="tm"><plugin instance="t"/></geom></worldbody></mujoco>""")
TORUS_ID = int(_probe.plugin[0])


# 2. The analytic SDF as a @wp.func pair (value, gradient).
@wp.func
def rounded_tile(p: wp.vec3, attr: vec_pluginattr) -> float:
    q = wp.vec3(wp.abs(p[0]) - attr[0], wp.abs(p[1]) - attr[0], wp.abs(p[2]))
    outside = wp.length(wp.max(q, wp.vec3(0.0)))
    inside = wp.min(wp.max(q[0], wp.max(q[1], q[2])), 0.0)
    return outside + inside - attr[1]


@wp.func
def rounded_tile_grad(p: wp.vec3, attr: vec_pluginattr) -> wp.vec3:
    q = wp.vec3(wp.abs(p[0]) - attr[0], wp.abs(p[1]) - attr[0], wp.abs(p[2]))
    g = wp.max(q, wp.vec3(0.0))
    g = wp.vec3(g[0] * wp.sign(p[0]), g[1] * wp.sign(p[1]), g[2] * wp.sign(p[2]))
    if wp.length(g) < 1e-9:                     # on the core square itself
        return wp.vec3(0.0, 0.0, 1.0)
    return wp.normalize(g)


# 3. Dispatchers with the placeholder signatures, specialised on the plugin id with wp.static.
@wp.func
def user_sdf(p: wp.vec3, attr: vec_pluginattr, sdf_type: int) -> float:
    if sdf_type == wp.static(TORUS_ID):
        return rounded_tile(p, attr)
    return 0.0


@wp.func
def user_sdf_grad(p: wp.vec3, attr: vec_pluginattr, sdf_type: int) -> wp.vec3:
    if sdf_type == wp.static(TORUS_ID):
        return rounded_tile_grad(p, attr)
    return wp.vec3(0.0)


# 4. Patch BEFORE any put_model / forward / step (same two assignments as register_sdf_plugins).
mjw._src.collision_sdf.user_sdf = user_sdf
mjw._src.collision_sdf.user_sdf_grad = user_sdf_grad

R, r, BALL = 0.2, 0.03, 0.05
XML = f"""
<mujoco model="custom_sdf">
  <extension><plugin plugin="mujoco.sdf.torus"><instance name="tile">
    <config key="radius1" value="{R}"/><config key="radius2" value="{r}"/>
  </instance></plugin></extension>
  <asset><mesh name="tile"><plugin instance="tile"/></mesh></asset>
  <option timestep="0.002" sdf_iterations="10" sdf_initpoints="40"/>
  <worldbody>
    <geom name="floor" type="plane" size="1 1 .1" pos="0 0 -0.3"/>
    <body name="tile" pos="0 0 0"><geom name="tile" type="sdf" mesh="tile">
      <plugin instance="tile"/></geom></body>
    <body name="ball" pos="0 0 {r + BALL + 0.02}"><freejoint/>
      <geom name="ball" type="sphere" size="{BALL}"/></body>
  </worldbody>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(XML)
print(f"torus plugin type id {TORUS_ID}; tile attrs R={R}, r={r}; geom_aabb half-size of tile "
      f"{mjm.geom_aabb[1, 3:]}")
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=64, njmax=256)
mjd = mujoco.MjData(mjm)
for _ in range(400):                       # 0.8 s
    mjw.step(m, d)
    mujoco.mj_step(mjm, mjd)
wp.synchronize()
nacon = min(int(d.nacon.numpy()[0]), d.naconmax)
dist = d.contact.dist.numpy()[:nacon]
nrm = d.contact.frame.numpy()[:nacon, 0]
print(f"MJWarp (patched tile): ball z = {float(d.qpos.numpy()[0, 2]):.4f}  "
      f"(expected r + radius = {r + BALL:.4f}), nacon {nacon}")
print(f"  contact dist min {dist.min():+.5f}, mean normal {nrm.mean(0)}")
print(f"MuJoCo C (real torus):  ball z = {mjd.qpos[2]:.4f}  (fell through the hole to the floor)")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

In [ ]:
!python ex8_custom_sdf.py

In [ ]:
# Level sets of the rounded tile in the x-z plane (numpy copy of the @wp.func, optional plot).
R, r = 0.2, 0.03
xs = np.linspace(-0.3, 0.3, 241); zs = np.linspace(-0.1, 0.1, 81)
X, Z = np.meshgrid(xs, zs)
q = np.stack([np.abs(X) - R, np.abs(Z)], -1)
phi = np.linalg.norm(np.maximum(q, 0), axis=-1) + np.minimum(q.max(-1), 0) - r
print('inside fraction of the window:', round(float((phi < 0).mean()), 3))
try:
    import matplotlib.pyplot as plt
    plt.contour(X, Z, phi, levels=np.linspace(-0.03, 0.06, 10)); plt.contour(X, Z, phi, [0], colors='k')
    plt.gca().set_aspect('equal'); plt.title('rounded tile: SDF level sets (x-z)'); plt.show()
except ImportError:
    print('skipped: matplotlib not installed')

## 9. Part IV checkpoint: falling clutter
TERRAIN + scatter(30): PRIMITIVE, CONVEX and HFIELD paths. Find the first overflow with small budgets, fix it, and count contacts per path.

In [ ]:
# Example 18.9 - ex9_checkpoint_clutter.py
"""Example 18.9 - Part IV checkpoint "Falling clutter": profile the rigid paths, fix overflow.

The scene is TERRAIN (model M7: a heightfield with a sphere, an ellipsoid, a cylinder and a
tetrahedron mesh) plus scatter(30) (model M9: 30 free spheres and capsules) over a floor plane.
It uses the PRIMITIVE, CONVEX and HFIELD narrowphase paths; no flex and no SDF.
1. Start from deliberately small budgets, step, and decode d.overflow (describe_overflow, Ch 9).
   Double whatever overflowed and try again until the run is clean.
2. With the fixed budgets, record every kernel that one mjw.collision call launches (we wrap
   wp.launch and keep the kernel names, as in Example 16.8) and count contacts per path.
Run: python ex9_checkpoint_clutter.py [--dump-json PATH]
"""
import argparse
import collections
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import warp_util                            # private API (kernel cache)
from mujoco_warp._src.collision_driver import MJ_COLLISION_TABLE  # private API

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 200
G = mjw.GeomType

TERRAIN_ASSETS = """
    <hfield name="bumps" nrow="5" ncol="5" size="1 1 .2 .1"
      elevation="0 0 0 0 0  0 .5 1 .5 0  0 1 0 1 0  0 .5 1 .5 0  0 0 0 0 0"/>
    <mesh name="tetra" vertex="0 0 0  .2 0 0  0 .2 0  0 0 .2"/>"""
TERRAIN_BODIES = """
    <geom name="ground" type="hfield" hfield="bumps"/>
    <body name="ball" pos="0.2 0.1 .6"><freejoint/><geom type="sphere" size=".08"/></body>
    <body name="ellipsoid" pos="-0.3 0.2 .6"><freejoint/>
      <geom type="ellipsoid" size=".1 .07 .05"/></body>
    <body name="cyl" pos="0.3 -0.3 .6"><freejoint/><geom type="cylinder" size=".06 .05"/></body>
    <body name="tetra" pos="-0.3 -0.3 .6"><freejoint/><geom type="mesh" mesh="tetra"/></body>"""


def scatter_bodies(n, seed=0, spacing=0.4, z0=0.9, per_side=10, center=(0.0, 0.0)):
    """The bodies of scatter(n) (model M9, Ch 15), started z0 above the floor.

    per_side bodies per row and per_side rows per layer, spacing apart, centred on center (x, y).
    """
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n):
        layer, k = divmod(i, per_side * per_side)
        row, col = divmod(k, per_side)
        x = center[0] - 0.5 * spacing * (per_side - 1) + spacing * col + rng.uniform(-0.02, 0.02)
        y = center[1] - 0.5 * spacing * (per_side - 1) + spacing * row + rng.uniform(-0.02, 0.02)
        z = z0 + spacing * layer
        if i % 2 == 0:
            geom = f'<geom type="sphere" size="{rng.uniform(0.05, 0.1):.3f}"/>'
        else:
            e = rng.uniform(-180, 180, size=3)
            geom = (f'<geom type="capsule" size="{rng.uniform(0.05, 0.07):.3f} '
                    f'{rng.uniform(0.05, 0.1):.3f}" euler="{e[0]:.0f} {e[1]:.0f} {e[2]:.0f}"/>')
        out.append(f'<body name="b{i}" pos="{x:.3f} {y:.3f} {z:.3f}"><freejoint/>{geom}</body>')
    return "".join(out)


# Drop the 30 bodies as a 5 x 5 (+5) block, 0.25 m apart, over the middle of the 2 m x 2 m
# heightfield: they land on the terrain and on the TERRAIN bodies, so the CONVEX pairs
# (sphere/capsule against ellipsoid, cylinder, mesh) and HFIELD-CAPSULE really make contacts.
LAYOUT = dict(spacing=0.25, per_side=5, z0=1.0, center=(0.0, 0.0))

CLUTTER = f"""
<mujoco model="falling_clutter">
  <option timestep="0.005"/>
  <asset>{TERRAIN_ASSETS}</asset>
  <worldbody>
    <light pos="0 0 4" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="3 3 .1"/>{TERRAIN_BODIES}
    {scatter_bodies(30, **LAYOUT)}
  </worldbody>
</mujoco>
"""


def describe_overflow(d, max_worlds=8):
    """Same as my_env.describe_overflow (Ch 9): "none", or the flagged bits per world."""
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    flagged = np.nonzero(ov)[0]
    lines = [f"world {w}: " + " | ".join(b.name for b in mjw.OverflowType if ov[w] & b)
             for w in flagged[:max_worlds]]
    counts = [f"{b.name} x{int(np.count_nonzero(ov & b))}" for b in mjw.OverflowType
              if (ov & b).any()]
    return "\n".join(lines + ["worlds per bit: " + ", ".join(counts)])


def path_of(t1, t2):
    if G.HFIELD in (t1, t2):
        return "HFIELD"
    return MJ_COLLISION_TABLE[(t1, t2)].name


mjm = mujoco.MjModel.from_xml_string(CLUTTER)
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0      # we decode d.overflow ourselves (this recompiles nothing yet)
types = sorted({mujoco.mjtGeom(t).name[7:] for t in mjm.geom_type})
print(f"nbody {mjm.nbody}  nv {mjm.nv}  ngeom {mjm.ngeom}  geom types {types}")
d1 = mjw.make_data(mjm, nworld=1)
print(f"io.py defaults: nconmax {d1.naconmax} per world, njmax {d1.njmax}")

# ---- 1. find and fix the first overflow ----
sizes = {"nconmax": 8, "njmax": 32, "nccdmax": 1}         # per world
FIX = {"BROADPHASE": "nconmax", "NARROWPHASE": "nconmax", "NEFC": "njmax", "CCD": "nccdmax"}
history = []
for attempt in range(6):
    d = mjw.make_data(mjm, nworld=NWORLD, **sizes)
    for _ in range(NSTEP):
        mjw.step(m, d)
    wp.synchronize()
    report = describe_overflow(d)
    bits = {b.name for b in mjw.OverflowType if (d.overflow.numpy() & b).any()}
    raw = int(d.nacon.numpy()[0])                       # can exceed naconmax
    print(f"try {attempt}: {sizes} nacon {raw}/{d.naconmax} -> {report.splitlines()[-1]}")
    history.append({"sizes": dict(sizes), "bits": sorted(bits), "nacon": raw,
                    "naconmax": d.naconmax,
                    "worldid": d.contact.worldid.numpy()[:min(raw, d.naconmax)].tolist()})
    if not bits:
        break
    grow = {FIX[b] for b in bits if b in FIX}
    if "nccdmax" in grow and sizes["nccdmax"] >= sizes["nconmax"]:
        grow.add("nconmax")                              # CCD is capped by nconmax
    for key in grow:
        sizes[key] *= 4 if key == "nccdmax" else 2
    sizes["nccdmax"] = min(sizes["nccdmax"], sizes["nconmax"])   # make_data: nccdmax <= nconmax

# ---- 2. one recorded collision call with the clean budgets ----
calls = []
_launch = wp.launch


built = {id(k): key for key, k in warp_util._KERNEL_CACHE.items()}   # kernel -> builder args


def label(kernel):
    key = built.get(id(kernel))
    if key and key[-1] in ("ccd_kernel_builder", "ccd_hfield_kernel_builder"):
        return f"{key[-1]}({G(key[0]).name}, {G(key[1]).name})"
    return kernel.key.split("__")[0]


def recording_launch(kernel, dim, *a, **kw):
    calls.append((label(kernel), np.atleast_1d(dim).tolist()))
    return _launch(kernel, dim, *a, **kw)


wp.launch = recording_launch
try:
    mjw.collision(m, d)
finally:
    wp.launch = _launch
wp.synchronize()
print("kernels launched by one mjw.collision call:")
for name, dim in calls:
    print(f"  {name:44s} dim {dim}")

nacon = min(int(d.nacon.numpy()[0]), d.naconmax)
gt = mjm.geom_type[d.contact.geom.numpy()[:nacon]]
per_path = collections.Counter(path_of(G(int(a)), G(int(b))) for a, b in gt)
per_pair = collections.Counter(f"{G(int(a)).name}-{G(int(b)).name}" for a, b in gt)
print(f"contacts after {NSTEP} steps, {NWORLD} worlds (nacon {nacon} of {d.naconmax}):")
for path in ["PRIMITIVE", "CONVEX", "HFIELD"]:
    pairs = {k: v for k, v in sorted(per_pair.items())
             if path_of(*[G[x] for x in k.split("-")]) == path}
    print(f"  {path:9s} {per_path[path]:4d}  {pairs}")
assert per_path["CONVEX"] > 0 and per_pair["HFIELD-CAPSULE"] > 0, "layout must exercise every path"
print(f"max nefc per world {int(d.nefc.numpy().max())} of njmax {d.njmax}")
print("overflow:", describe_overflow(d))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"history": history, "per_path": dict(per_path), "per_pair": dict(per_pair),
                   "calls": calls, "nworld": NWORLD, "nstep": NSTEP}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## Exercise 18.1: tune the heightfield resolution
Find the finest grid on which a box of half-size 0.15 m rests without the HFIELD overflow bit.

In [ ]:
# Exercise 18.1: the finest heightfield grid a 0.15 m box can sit on without HFIELD overflow.
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HALF = 0.15          # box half-size (bigger than in Example 18.4)


def hfield_overflow(n, half=HALF):
    """Return the HFIELD overflow bit after one mjw.forward for a box resting on an n x n grid."""
    ...                 # build the scene of Example 18.4 with box size `half`, forward, read d.overflow


N = ...                 # the largest n (5..41) with no HFIELD overflow
assert hfield_overflow(N) == 0, "still overflowing"
assert all(hfield_overflow(k) for k in range(N + 1, 42)), "you can go finer"
print("ok")

In [ ]:
#@title Exercise 18.1: solution (double-click to show the code)
HALF = 0.15
from functools import cache


@cache                  # the checks below ask for the same n again
def hfield_overflow(n, half=HALF):
    flat = " ".join(["0"] * (n * n))
    xml = f"""<mujoco><asset><hfield name="f" nrow="{n}" ncol="{n}" size="0.5 0.5 0.1 0.05"
      elevation="{flat}"/></asset><worldbody><geom type="hfield" hfield="f"/>
      <body pos="0.013 0.007 {half - 0.002}"><freejoint/>
      <geom type="box" size="{half} {half} {half}"/></body></worldbody></mujoco>"""
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    d = mjw.make_data(mjm, nworld=1, nconmax=16, njmax=64)
    mjw.forward(m, d)
    return int(d.overflow.numpy()[0]) & mjw.OverflowType.HFIELD


# scan from fine to coarse and keep the first clean resolution; overflow is not monotonic
# in n (n = 15 overflows, n = 16 does not), so a coarse-to-fine scan would stop too early
N = next(n for n in range(41, 4, -1) if not hfield_overflow(n))
print("finest clean grid:", N)
assert hfield_overflow(N) == 0, "still overflowing"
assert all(hfield_overflow(k) for k in range(N + 1, 42)), "you can go finer"
print("ok")

## Exercise 18.2: a sphere SDF through the user hook
Fill in the TODOs, then run the next cell (fresh process). Hint: for two overlapping spheres, `max(A, B)` is lowest at the middle of the overlap segment on the centre line.

In [ ]:
%%writefile ex18_2_sphere_sdf.py
# Exercise 18.2: a sphere SDF through the user hook (runs in a fresh process: see the next cell).
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec_pluginattr

probe = mujoco.MjModel.from_xml_string("""<mujoco><extension>
  <plugin plugin="mujoco.sdf.torus"><instance name="t"/></plugin></extension>
  <asset><mesh name="tm"><plugin instance="t"/></mesh></asset>
  <worldbody><geom type="sdf" mesh="tm"><plugin instance="t"/></geom></worldbody></mujoco>""")
TORUS_ID = int(probe.plugin[0])

# TODO 1: write @wp.func user_sdf(p, attr, sdf_type) -> float: a sphere of radius attr[1]
# TODO 2: write @wp.func user_sdf_grad(p, attr, sdf_type) -> wp.vec3
# TODO 3: assign both to mjw._src.collision_sdf BEFORE any MJWarp call

A, B, C = 0.1, 0.05, 0.145            # SDF sphere radius, ball radius, centre distance
xml = f"""<mujoco><extension><plugin plugin="mujoco.sdf.torus"><instance name="s">
  <config key="radius1" value="0.0001"/><config key="radius2" value="{A}"/></instance></plugin>
  </extension><asset><mesh name="s"><plugin instance="s"/></mesh></asset><worldbody>
  <body><geom type="sdf" mesh="s"><plugin instance="s"/></geom></body>
  <body pos="0 0 {C}"><freejoint/><geom type="sphere" size="{B}"/></body>
  </worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(xml)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=64, njmax=256)
mjw.kinematics(m, d)
mjw.collision(m, d)
n = int(d.nacon.numpy()[0])
dist = d.contact.dist.numpy()[:n]
normal = d.contact.frame.numpy()[:n, 0]
best = ...                            # TODO 4: the lowest value max(A, B) can take (analytic)
assert n > 0, "no contact: was the patch applied before the first MJWarp call?"
assert dist.min() >= best - 1e-4, "deeper than the analytic minimum of max(A, B)"
assert dist.max() < 0.0 and dist.min() < best / 2, "contact too shallow"
assert abs(normal[np.argmin(dist), 2]) > 0.9, "deepest normal should follow the centre line"
print("ok")

In [ ]:
!python ex18_2_sphere_sdf.py

In [ ]:
#@title Exercise 18.2: solution (double-click to show the code)
SRC = r'''import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec_pluginattr

probe = mujoco.MjModel.from_xml_string("""<mujoco><extension>
  <plugin plugin="mujoco.sdf.torus"><instance name="t"/></plugin></extension>
  <asset><mesh name="tm"><plugin instance="t"/></mesh></asset>
  <worldbody><geom type="sdf" mesh="tm"><plugin instance="t"/></geom></worldbody></mujoco>""")
TORUS_ID = int(probe.plugin[0])


@wp.func
def user_sdf(p: wp.vec3, attr: vec_pluginattr, sdf_type: int) -> float:
    if sdf_type == wp.static(TORUS_ID):
        return wp.length(p) - attr[1]          # sphere of radius radius2
    return 0.0


@wp.func
def user_sdf_grad(p: wp.vec3, attr: vec_pluginattr, sdf_type: int) -> wp.vec3:
    if sdf_type == wp.static(TORUS_ID) and wp.length(p) > 1e-9:
        return wp.normalize(p)
    return wp.vec3(0.0, 0.0, 1.0)


mjw._src.collision_sdf.user_sdf = user_sdf          # BEFORE any MJWarp call
mjw._src.collision_sdf.user_sdf_grad = user_sdf_grad

A, B, C = 0.1, 0.05, 0.145            # SDF sphere radius, ball radius, centre distance
xml = f"""<mujoco><extension><plugin plugin="mujoco.sdf.torus"><instance name="s">
  <config key="radius1" value="0.0001"/><config key="radius2" value="{A}"/></instance></plugin>
  </extension><asset><mesh name="s"><plugin instance="s"/></mesh></asset><worldbody>
  <body><geom type="sdf" mesh="s"><plugin instance="s"/></geom></body>
  <body pos="0 0 {C}"><freejoint/><geom type="sphere" size="{B}"/></body>
  </worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(xml)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=64, njmax=256)
mjw.kinematics(m, d)
mjw.collision(m, d)
n = int(d.nacon.numpy()[0])
dist = d.contact.dist.numpy()[:n]
normal = d.contact.frame.numpy()[:n, 0]
overlap = A + B - C                     # analytic: the lens is 5 mm deep along the centre line
best = -overlap / 2                     # the lowest possible max(A, B), at the lens midpoint
print(f"nacon {n}, min dist {dist.min():+.5f}, analytic bound {best:+.5f}")
assert n > 0, "no contact: was the patch applied before the first MJWarp call?"
assert dist.min() >= best - 1e-4, "deeper than the analytic minimum of max(A, B)"
assert dist.max() < 0.0 and dist.min() < best / 2, "contact too shallow"
assert abs(normal[np.argmin(dist), 2]) > 0.9, "deepest normal should follow the centre line"
print("ok")
'''
open("ex18_2_solution.py", "w").write(SRC)
import subprocess, sys
run = subprocess.run([sys.executable, "ex18_2_solution.py"], capture_output=True, text=True)
print(run.stdout or run.stderr)

## Stretch: a torus SDF
Write `torus(p, attr)` and its gradient yourself (`attr[0]` major radius, `attr[1]` minor radius), patch it in a fresh process, and drop a capsule through the hole. Compare with `mujoco_warp/test_data/collision_sdf/torus.py`.

## Part IV review (Ch 15-18)
Answer for yourself, then check against the chapter page (the quiz there gives instant feedback).
1. Which stage turns all geom pairs into a short candidate list, and where does it write it? *(the broadphase, NXN or SAP, into the collision context's pair list, counted by `d.ncollision`)*
2. For a pair (BOX, SPHERE) listed in the XML as sphere first, which way does `contact.frame[0]` point? *(from `geom[0]` to `geom[1]`, where `geom[0]` is the lower type: from the sphere (2) to the box (6))*
3. "CCD overflow - please increase naccdmax". What ran out? *(the GJK/EPA scratch slots of one convex or HFIELD pair type; each kernel reserves one with `atomic_add(nccd, geomgeomid, 1)`)*
4. Which geom pair has no collision function at all in MJWarp 4.0.1? *(hfield-plane: plane-plane, plane-hfield and hfield-hfield are not in `MJ_COLLISION_TABLE`)*
5. Why does GJK need a support function, and which shapes cannot provide one? *(it explores the Minkowski difference through extreme points; non-convex shapes such as terrain and SDFs have no useful one, hence prisms and gradient descent)*
6. After a long RL rollout `d.overflow` shows NARROWPHASE, but `nacon < naconmax` now. Is the current step fine? *(maybe: the bits are sticky and record that some step overflowed; zero `d.overflow` to monitor per step, as Example 18.9 shows)*